# 🧰 Output Parsers in LangChain

## Learning Objectives
**Output parsers** transform unstructured LLM text into structured Python objects. In this notebook, you'll learn:
1. **PydanticOutputParser** - parse LLM output into validated Pydantic models
2. **JsonOutputParser** - parse into Python dictionaries, and run one chain over several inputs
3. **CommaSeparatedListOutputParser** - parse into Python lists

## Where this fits
`03_Prompt_Templates.ipynb` and `04_Few_Shot_Composition_and_Serialization.ipynb` shaped what goes
*into* the model. This notebook shapes what comes *out*. The next module,
`modules/03_lcel_and_runnables/`, composes prompts, models and parsers into larger chains.

## Prerequisites
- `03_Prompt_Templates.ipynb` (reading)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

## 💡 Why Use Output Parsers?

| Without Parser | With Parser |
|---------------|-------------|
| `"The sentiment is positive and the topic is AI"` | `{"sentiment": "positive", "topic": "AI"}` |
| Manual string parsing | Automatic type validation |
| Error-prone | Reliable and consistent |

## 🔑 Key Concept

Output parsers work in two steps:
1. **Inject format instructions** into the prompt
2. **Parse the LLM response** into structured data

---
## 📦 Step 1: Environment Setup

Load the API key from the project-root `.env`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: the model that fills the structured responses
# ============================================================================
from aied_helpers import get_openai_llm

# More capable models generally produce better structured outputs
llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---

## 📊 Overview of Output Parsers

LangChain provides several output parsers for different use cases:

| Parser | Output Type | Best For |
|--------|-------------|----------|
| `PydanticOutputParser` | Pydantic Model | Complex schemas with validation |
| `JsonOutputParser` | Python Dict | Simple JSON structures |
| `CommaSeparatedListOutputParser` | Python List | Lists of items |
| `StrOutputParser` | String | Simple text extraction |

---
## 🔷 Part 1: PydanticOutputParser

The most powerful parser - converts LLM output into **validated Pydantic models**.

**Advantages:**
- Type checking and validation
- IDE autocomplete support
- Clear schema definition
- Automatic error messages

> **Note:** Use a capable LLM (GPT-4o-mini or better) for reliable structured output.

In [ ]:
# ============================================================================
# STEP 1: DEFINE YOUR DATA MODEL
# ============================================================================
# Pydantic models define the structure of your expected output
# Field() adds descriptions that help the LLM understand what to generate
# ============================================================================

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# Define the output structure using Pydantic
class QueryResponse(BaseModel):
    """Structured response for topic analysis"""
    description: str = Field(description="A brief description of the topic asked by the user")
    pros: str = Field(description="3 bullet points showing the pros of the topic")
    cons: str = Field(description="3 bullet points showing the cons of the topic")
    conclusion: str = Field(description="One line conclusion of the topic")

# Create the parser from the Pydantic model
parser = PydanticOutputParser(pydantic_object=QueryResponse)

print("✅ PydanticOutputParser created")
print(f"📋 Expected fields: {list(QueryResponse.model_fields.keys())}")

In [ ]:
# ============================================================================
# STEP 2: VIEW FORMAT INSTRUCTIONS
# ============================================================================
# The parser auto-generates instructions that tell the LLM how to format output
# These instructions are injected into your prompt!
# ============================================================================

print("📜 Format Instructions (injected into prompt):")
print("=" * 60)
print(parser.get_format_instructions())
print("=" * 60)

In [ ]:
# ============================================================================
# STEP 3: CREATE THE CHAIN
# ============================================================================
# The chain: Prompt → LLM → Parser
# - partial_variables injects the format instructions automatically
# - The parser converts the LLM's text output into a Pydantic object
# ============================================================================

prompt_txt = """
Answer the user query and generate the response based on the following formatting instructions.

Format Instructions:
{format_instructions}

Query:
{query}
"""

prompt = PromptTemplate(
    template=prompt_txt,
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

# Build the LCEL chain
chain = prompt | llm | parser

# Test with a simple query
response = chain.invoke({"query": "What is the capital of France?"})

print("🎯 Parsed Response:")
print(f"   Type: {type(response)}")
print(f"   Description: {response.description}")

In [ ]:
# ============================================================================
# USING THE PARSED RESPONSE: fields are Python attributes
# ============================================================================
question = "Tell me about Commercial Real Estate"
response = chain.invoke({"query": question})

print(f"📝 Query: {question}")
print("-" * 50)
print("📋 Description:")
print(response.description)
print("\n✅ Pros:")
print(response.pros)

In [ ]:
# ============================================================================
# CONVERTING TO A DICTIONARY: model_dump() for JSON serialization
# ============================================================================
print("📋 All Fields:")
print("=" * 50)
for k, v in response.model_dump().items():
    print(f"\n🔹 {k.upper()}:")
    print(f"   {v}")

---
## 🔷 Part 2: JsonOutputParser

Similar to PydanticOutputParser but returns a **Python dictionary** instead of a Pydantic model.

**When to use:**
- When you need a simple dict, not a Pydantic model
- For dynamic schemas
- When you want flexibility over strict typing

In [ ]:
# ============================================================================
# JSONOUTPUTPARSER: same schema, dict output
# ============================================================================
# Reuses the QueryResponse model from Part 1 - the schema only shapes the
# format instructions; the parsed result is a plain dict
# ============================================================================
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser(pydantic_object=QueryResponse)

print(f"✅ {type(parser).__name__} created for {QueryResponse.__name__}")

In [ ]:
# ============================================================================
# JSON PARSER PROMPT: same template shape, format instructions from the new parser
# ============================================================================
prompt_txt = """
Answer the user query and generate the response based on the following formatting instructions.

Format Instructions:
{format_instructions}

Query:
{query}
"""
prompt = PromptTemplate(
    template=prompt_txt,
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

print(f"📋 Input variables: {prompt.input_variables}")
print(f"📋 Pre-filled     : {list(prompt.partial_variables)}")

In [ ]:
# ============================================================================
# CHAIN + BATCH: one chain, several queries
# ============================================================================
chain = prompt | llm | parser

topic_queries = [
    "Tell me about commercial real estate",
    "Tell me about Generative AI"
]
topic_queries_formatted = [{"query": topic} for topic in topic_queries]

responses = chain.batch(topic_queries_formatted)

print(f"📊 {len(responses)} responses, first one is a {type(responses[0]).__name__}")
for response in responses:
    for k, v in response.items():
        print(f"{k}:\n{v}\n")
    print("-----")

In [ ]:
# ============================================================================
# DICTS TO A TABLE: list of dicts -> pandas DataFrame
# ============================================================================
import pandas as pd

df = pd.DataFrame(responses)
print(df.to_string(max_colwidth=50))

---
## 🔷 Part 3: CommaSeparatedListOutputParser

The simplest parser - converts comma-separated text into a **Python list**.

**When to use:**
- Generating lists of items
- Simple enumerations
- Tags or categories

In [ ]:
# ============================================================================
# COMMASEPARATEDLISTOUTPUTPARSER
# ============================================================================

from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate

output_parser = CommaSeparatedListOutputParser()

# View the format instructions
format_instructions = output_parser.get_format_instructions()
print("📜 Format Instructions:")
print(format_instructions)

In [ ]:
# ============================================================================
# LIST PROMPT: no user variables, only the pre-filled format instructions
# ============================================================================
prompt_txt = """
Create a list of 5 different ways in which Generative AI can be used

Output format instructions:
{format_instructions}
"""

prompt = PromptTemplate(
    template=prompt_txt,
    partial_variables={"format_instructions": format_instructions},
)
print(f"📋 Input variables: {prompt.input_variables}")  # [] - nothing left to fill

In [ ]:
# ============================================================================
# RUN THE CHAIN: an empty dict, because every variable is pre-filled
# ============================================================================
llm_chain = prompt | llm | output_parser
response = llm_chain.invoke({})

print("📋 Parsed List:")
print(response)
print(f"\n📊 Type: {type(response).__name__}")

print("\n📝 List Items:")
for i, item in enumerate(response, 1):
    print(f"  {i}. {item}")

---
## 📝 Summary

In this notebook, we learned:

### 1. Why Output Parsers Matter
- Raw LLM output is unstructured text — output parsers convert it into structured Python objects your application can use directly
- Every parser works the same two-step way: inject **format instructions** into the prompt, then **parse** the LLM's response

### 2. PydanticOutputParser
- Converts LLM output into a **validated Pydantic model**, giving type checking, IDE autocomplete, and automatic error messages
- Best for complex schemas where you want strict validation

### 3. JsonOutputParser
- Same format-instruction pattern as `PydanticOutputParser`, but returns a plain **Python dict**
- `chain.batch([...])` runs one chain over several inputs
- Useful when you want the flexibility of a dict without strict Pydantic typing

### 4. CommaSeparatedListOutputParser
- The simplest parser — turns a comma-separated response into a **Python list**
- Great for tags, categories, or enumerated lists

### Key Pattern
```python
chain = prompt | llm | parser
response = chain.invoke({"query": "..."})
```
The parser slots in as the last step of an LCEL chain, right after the LLM.

### Next Steps
- **`modules/03_lcel_and_runnables/notebooks/01_LCEL_Introduction.ipynb`**: prompts, models and parsers composed into richer LCEL chains